# UmFreela Data Lakehouse: execução do pipeline no Google Colab

**Disciplina:** Introdução ao Big Data (Prof. Felipe Becker Nunes) · **Aluno:** Jonathan Gonçalves Possa

Este notebook executa o pipeline completo e mostra as **evidências de execução** pedidas no enunciado: lotes ingeridos, camadas Bronze, Silver e Gold, orquestração, logs, consultas, validações e dashboard.

Basta executar as células em ordem (**Ambiente de execução > Executar tudo**). Não é preciso GPU.

## 1. Preparar o ambiente
Clona o repositório e instala as dependências. O Colab já tem Python e Java (exigido pelo Spark).

In [ ]:
REPO_URL = "https://github.com/Jonathanpossa/umfreela-data-lakehouse.git"
import os
if not os.path.exists("/content/umfreela-data-lakehouse"):
    !git clone -q $REPO_URL /content/umfreela-data-lakehouse
%cd /content/umfreela-data-lakehouse
!pip -q install -r requirements.txt
!java -version 2>&1 | head -1
import pyspark, duckdb
print("PySpark", pyspark.__version__, "| DuckDB", duckdb.__version__)

## 2. Primeira execução: 335 dias de operação

O orquestrador executa as 7 etapas em sequência: gerar dados, ingestão Bronze, Silver, Gold, validações, consultas e dashboard. Se uma etapa falhar, as seguintes não rodam.

In [ ]:
!python orchestration/run_pipeline.py --dias 335

### Evidência: lotes que chegaram na landing zone
Cada arquivo representa um dia de operação da plataforma (JSON Lines). Note os problemas de qualidade: formatos de data diferentes, cidades escritas de jeitos diferentes etc.

In [ ]:
import duckdb, pandas as pd, json, glob
pd.set_option("display.max_colwidth", 120)
arquivos = sorted(glob.glob("data/generated/eventos/*.jsonl"))
print(len(arquivos), "lotes na landing | primeiro:", arquivos[0].split("/")[-1], "| último:", arquivos[-1].split("/")[-1])
!head -n 5 data/generated/eventos/eventos_2026-05-15.jsonl

### Evidência: camada Bronze
O evento fica exatamente como chegou (`json_original`), com o arquivo de origem e o horário da ingestão.

In [ ]:
con = duckdb.connect()
BRONZE = "read_parquet('lake/bronze/eventos/**/*.parquet', hive_partitioning=true)"
display(con.execute(f"SELECT arquivo_origem, data_lote, ingestao_ts, json_original FROM {BRONZE} LIMIT 5").df())
display(con.execute(f"SELECT COUNT(*) AS eventos_na_bronze, COUNT(DISTINCT data_lote) AS lotes FROM {BRONZE}").df())

### Evidência: camada Silver
Eventos limpos, com datas convertidas, cidade e função padronizadas e a cobrança calculada (R$ 3,99 por vaga, grátis nos 15 primeiros dias do restaurante). Abaixo, os registros rejeitados e o motivo.

In [ ]:
SILVER = "read_parquet('lake/silver/eventos/**/*.parquet', hive_partitioning=true)"
display(con.execute(f"""SELECT event_id, tipo_evento, event_ts, id_vaga, cidade, funcao, valor_diaria,
                                 em_periodo_gratuito, valor_cobrado
                          FROM {SILVER} WHERE tipo_evento = 'vaga_publicada' LIMIT 5""").df())
display(con.execute("""SELECT event_id, tipo_evento, cidade_original, arquivo_origem, motivo_rejeicao
                         FROM read_parquet('lake/silver/eventos_rejeitados/*.parquet')
                         WHERE motivo_rejeicao <> 'duplicado' LIMIT 8""").df())

### Evidência: camada Gold
Indicadores mensais calculados a partir da tabela `fato_vagas`.

In [ ]:
display(con.execute("SELECT * FROM read_parquet('lake/gold/fato_vagas/*.parquet') LIMIT 5").df())
display(con.execute("SELECT * FROM read_parquet('lake/gold/kpi_mensal/*.parquet') ORDER BY mes").df())

## 3. Nova execução: chegam mais 30 dias (ingestão incremental)

Agora a landing recebe os dias 336 a 365. A ingestão lê **apenas os 30 lotes novos**, porque consulta o arquivo de controle dos lotes já ingeridos.

In [ ]:
!python orchestration/run_pipeline.py --dias 365

### Evidência: orquestração, logs e monitoramento
Cada execução registra o status e a duração de cada etapa.

In [ ]:
display(pd.read_csv("lake/monitoramento/execucoes.csv"))
print("\n--- últimas linhas de logs/pipeline.log ---")
!tail -n 25 logs/pipeline.log

## 4. Consultas analíticas (DuckDB + SQL)
As consultas estão em `queries/consultas.sql`.

In [ ]:
import sys; sys.path.append(".")
from queries.executar_consultas import executar
resultados = executar(mostrar=False)
for nome, df in resultados.items():
    print("\n=====", nome, "=====")
    display(df)

## 5. Validações de qualidade (pytest)

In [ ]:
!python -m pytest -v tests/

## 6. Dashboard

In [ ]:
from IPython.display import Image
Image("dashboard/dashboard_umfreela.png")